In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import DataLoader
from src.indicators import smc_custom
import plotly.graph_objects as go

symbol = "TSLA"
mid_tf = "5min"

loader = DataLoader(symbol=symbol)
df_mid = loader.get_data(mid_tf, force_refresh=False)
df_mid = df_mid[:df_mid.shape[0] // 10]
inflexions_mid = smc_custom.inflexion_points(df_mid)
bos_mid = smc_custom.bos(df_mid, inflexions_mid, close_break=True)
ob_mid = smc_custom.ob(df_mid, bos_mid, inflexions_mid)

📂 Loaded 5000 candles from cache: /Users/nikolastsalidis/Desktop/smart-money-concepts/data/tsla/tsla_5min.csv


In [6]:
import numpy as np

fig = go.Figure(data=[go.Candlestick(
    x=df_mid.index,
    open=df_mid["open"],
    high=df_mid["high"],
    low=df_mid["low"],
    close=df_mid["close"],
    increasing_line_color="#77dd76",
    decreasing_line_color="#ff6962",
    name="TSLA",
)])

# Add OB zones (EndIndex now = invalidation candle or end of data)
ob_rows = ob_mid[ob_mid["OB"].notna()]
for _, row in ob_rows.iterrows():
    is_bull = row["OB"] == 1
    color = "rgba(0,188,212,0.2)" if is_bull else "rgba(255,87,34,0.2)"
    border = "cyan" if is_bull else "orange"
    x0 = int(row["StartIndex"])
    x1 = int(row["EndIndex"])
    fig.add_shape(
        type="rect", x0=x0, x1=x1,
        y0=row["Bottom"], y1=row["Top"],
        fillcolor=color, line=dict(color=border, width=1),
        layer="below",
    )

fig.update_layout(
    template="plotly_dark",
    title="TSLA - Order Blocks",
    xaxis_rangeslider_visible=False,
    height=700,
)
fig.show()

In [ ]:
bull_obs = ob_rows[ob_rows["OB"] == 1]
bear_obs = ob_rows[ob_rows["OB"] == -1]
zone_sizes = ob_rows["Top"] - ob_rows["Bottom"]

print(f"Total OBs: {len(ob_rows)}  |  Bullish: {len(bull_obs)}  |  Bearish: {len(bear_obs)}")
print(f"Avg zone size: ${zone_sizes.mean():.2f}  |  Median: ${zone_sizes.median():.2f}")
print(f"Min: ${zone_sizes.min():.2f}  |  Max: ${zone_sizes.max():.2f}")